# fig1_new_region - Part 6/8\n\n自动拆分版本（按步骤执行）。\n包含统一 bootstrap 和共享模块导入。\n

In [ ]:
# AUTO_BOOTSTRAP_V2
from pathlib import Path
import sys
import os
import builtins
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "projects").exists():
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "projects").exists():
            ROOT = p
            break

NB_PATH = Path.cwd()
if "clone_motif" in str(NB_PATH):
    PROJECT_DIR = ROOT / "projects" / "clone_motif"
else:
    PROJECT_DIR = ROOT / "projects" / "our_multiregion_motif"

DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
OUT_FIG = PROJECT_DIR / "outputs" / "figures"
OUT_TABLE = PROJECT_DIR / "outputs" / "tables"
OUT_ARCH = PROJECT_DIR / "outputs" / "archives"

for d in [DATA_PROCESSED, OUT_FIG, OUT_TABLE, OUT_ARCH]:
    d.mkdir(parents=True, exist_ok=True)

SHARED_SRC = ROOT / "projects" / "shared" / "src"
if str(SHARED_SRC) not in sys.path:
    sys.path.append(str(SHARED_SRC))

from motif_common import combination, indices_for_region, union_indices_for_regions, p_to_star, format_p_decimal_3sig, sort_by_order, truncate_colormap

READ_EXT = {".csv", ".json", ".npy", ".pkl", ".xlsx"}
FIG_EXT = {".svg", ".png", ".pdf"}
TABLE_EXT = {".csv", ".xlsx"}


def _as_path(x):
    return Path(x) if isinstance(x, (str, os.PathLike)) else x


def resolve_read_path(path):
    p = _as_path(path)
    if not isinstance(p, Path):
        return path
    if p.is_absolute() or p.exists():
        return str(p)
    if p.suffix.lower() in READ_EXT:
        for c in [DATA_RAW / p.name, ROOT / p.name]:
            if c.exists():
                return str(c)
    return str(p)


def resolve_write_path(path):
    p = _as_path(path)
    if not isinstance(p, Path):
        return path
    if p.is_absolute():
        p.parent.mkdir(parents=True, exist_ok=True)
        return str(p)
    ext = p.suffix.lower()
    if ext in FIG_EXT:
        out = OUT_FIG / p.name
    elif ext in TABLE_EXT:
        out = OUT_TABLE / p.name
    elif ext == ".zip":
        out = OUT_ARCH / p.name
    elif ext == ".npy":
        out = DATA_PROCESSED / p.name
    else:
        out = PROJECT_DIR / p
    out.parent.mkdir(parents=True, exist_ok=True)
    return str(out)

if not hasattr(builtins, "_orig_open_codex"):
    builtins._orig_open_codex = builtins.open


def _open_patch(file, mode="r", *args, **kwargs):
    if isinstance(file, (str, os.PathLike)):
        if any(m in mode for m in ["r", "a"]):
            file = resolve_read_path(file)
        if any(m in mode for m in ["w", "a", "x"]):
            file = resolve_write_path(file)
    return builtins._orig_open_codex(file, mode, *args, **kwargs)


builtins.open = _open_patch

if not hasattr(np, "_orig_load_codex"):
    np._orig_load_codex = np.load
np.load = lambda file, *a, **k: np._orig_load_codex(resolve_read_path(file), *a, **k)

if not hasattr(np, "_orig_save_codex"):
    np._orig_save_codex = np.save
np.save = lambda file, arr, *a, **k: np._orig_save_codex(resolve_write_path(file), arr, *a, **k)

if not hasattr(pd, "_orig_read_csv_codex"):
    pd._orig_read_csv_codex = pd.read_csv
pd.read_csv = lambda f, *a, **k: pd._orig_read_csv_codex(resolve_read_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)

if not hasattr(pd, "_orig_read_excel_codex"):
    pd._orig_read_excel_codex = pd.read_excel
pd.read_excel = lambda f, *a, **k: pd._orig_read_excel_codex(resolve_read_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)

if not hasattr(pd.DataFrame, "_orig_to_csv_codex"):
    pd.DataFrame._orig_to_csv_codex = pd.DataFrame.to_csv


def _to_csv_patch(self, path_or_buf=None, *args, **kwargs):
    if isinstance(path_or_buf, (str, os.PathLike)):
        path_or_buf = resolve_write_path(path_or_buf)
    return pd.DataFrame._orig_to_csv_codex(self, path_or_buf, *args, **kwargs)


pd.DataFrame.to_csv = _to_csv_patch

if not hasattr(pd.DataFrame, "_orig_to_excel_codex"):
    pd.DataFrame._orig_to_excel_codex = pd.DataFrame.to_excel


def _to_excel_patch(self, excel_writer, *args, **kwargs):
    if isinstance(excel_writer, (str, os.PathLike)):
        excel_writer = resolve_write_path(excel_writer)
    return pd.DataFrame._orig_to_excel_codex(self, excel_writer, *args, **kwargs)


pd.DataFrame.to_excel = _to_excel_patch

try:
    import matplotlib.pyplot as plt
    if not hasattr(plt, "_orig_savefig_codex"):
        plt._orig_savefig_codex = plt.savefig
    plt.savefig = lambda f, *a, **k: plt._orig_savefig_codex(resolve_write_path(f) if isinstance(f, (str, os.PathLike)) else f, *a, **k)
except Exception:
    pass

print(f"[bootstrap] project={PROJECT_DIR.name} data={DATA_RAW}")


In [29]:
import pickle

files = [
    "wb_alltype_sc_results_dict_1115_pl.pkl",
    "wb_alltype_sc_results_dict_1115_ssp.pkl",
    "wb_alltype_sc_results_dict_1110.pkl",
]

merged = {"results": {}, "invalid_regions": set(), "error_regions": set(), "region_list": set()}

for path in files:
    with open(path, "rb") as f:
        p = pickle.load(f)
    for thr, regs in p["results"].items():
        merged["results"].setdefault(thr, {}).update(regs)
    merged["invalid_regions"].update(p.get("invalid_regions", []))
    merged["error_regions"].update(p.get("error_regions", []))
    merged["region_list"].update(p.get("region_list", []))

for k in ["invalid_regions", "error_regions", "region_list"]:
    merged[k] = sorted(merged[k])

with open("wb_alltype_sc_results_dict_merged.pkl", "wb") as f:
    pickle.dump(merged, f)

FileNotFoundError: [Errno 2] No such file or directory: 'wb_alltype_sc_results_dict_1115_pl.pkl'

In [ ]:
import pickle, numpy as np, torch

# 读 SC & ER 结果
with open("wb_alltype_sc_results_dict_merged.pkl", "rb") as f:
    sc_pack = pickle.load(f)
res_sc = sc_pack["results"]          # res_sc[thr][region]["sc"]

with open("er_sampling_results_all_region.pkl", "rb") as f:
    er_res = pickle.load(f)          # er_res[region]["er_mu"/"er_sd"]

regions       = sorted(res_sc[5].keys())
threshold_set = [5]
motif_results = {}

for thr in threshold_set:
    for region in regions:
        info = res_sc.get(thr, {}).get(region)
        er   = er_res.get(region)
        if info is None or er is None:
            print(f"[WARN] skip {region}, thr={thr} (no SC/ER)")
            continue

        sc = np.asarray(info["sc"], float)
        N  = sc.shape[0]
        if N <= 1:
            print(f"[WARN] {region}, thr={thr}: N={N}, skip.")
            continue

        np.fill_diagonal(sc, 0.0)
        A = (sc > 0).astype(int)
        E = int(A.sum())
        density = E / (N * (N - 1))

        mr = motifRegular(numOfNeuron=N)
        real = mr.cal(torch.from_numpy(A)).cpu().numpy().astype(float).ravel()
        # if real.shape[0] == 14: real = real[1:]
        if real.shape[0] != 13: raise ValueError(f"{region}, thr={thr}: motif_count len={real.shape[0]}")

        mu  = np.asarray(er["er_mu"], float)
        std = np.asarray(er["er_sd"], float) + 1
        if mu.shape[0] != 13 or std.shape[0] != 13:
            raise ValueError(f"{region}: ER mu/std len != 13")

        std_safe = np.where(std == 0, np.inf, std)
        Z = (real - mu) / std_safe
        Z[~np.isfinite(Z)] = 0.0
        nz_norm = np.linalg.norm(Z)
        NZ = Z / nz_norm if nz_norm > 0 else np.zeros_like(Z)

        motif_results.setdefault(region, {})[thr] = {
            "N": N, "E": E, "density": density,
            "motif_count": real, "er_mu": mu, "er_std": std,
            "Z": Z, "NZ": NZ,
        }
        print(f"[OK] {region}, thr={thr}, N={N}, E={E}, density={density:.4g}")

# 如需保存，取消注释：
# with open("motif_results_by_region_from_sc.pkl", "wb") as f:
#     pickle.dump(motif_results, f)

In [ ]:
with open("motif_results_by_region_from_sc.pkl", "wb") as f:
    pickle.dump(motif_results, f)

In [ ]:
thr = 5
total_N_thr5 = sum(
    motif_results[region][thr]["N"]
    for region in motif_results
    if thr in motif_results[region]
)
print(total_N_thr5)

In [ ]:
len([motif_results[region][thr]["N"]
    for region in motif_results])

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ===== 1. 读取 motif 结果 =====
with open("motif_results_by_region_5um_from_sc.pkl", "rb") as f:
    pack = pickle.load(f)

# threshold_set = pack.get("threshold_set", None)

# 你要用哪个 threshold（如果 motif_results 里有 threshold 维度的话）
# 如果只算了一个阈值，可以直接写那个值；或者 threshold_set[0]
thr_to_use = 5   # 根据你实际的 threshold_set 改

# ===== 2. 你给的脑区列表 =====
isocortex = [
    "FRP",
    "MOp", "MOs",
    "SSp", "SSs",
    "GU", "VISC",
    "AUDd", "AUDp", "AUDpo", "AUDv",
    "VISal", "VISam", "VISl", "VISp", "VISpl", "VISpm", "VISli", "VISpor",
    "ACAd", "ACAv", "PL", "ILA",
    "ORBl", "ORBm", "ORBvl",
    "AId", "AIp", "AIv",
    "RSPagl", "RSPd", "RSPv",
    "PTLp",
    "VISa", "VISrl", "TEa", "PERI", "ECT",
]

# --- HPF: Hippocampal formation（海马形成）---

# HIP: Hippocampal region 下到 area / region 这一层
hpf_hip_areas = [
    "CA1",
    "CA2",
    "CA3",   # Ammon's horn（下有 CA1/CA2/CA3，就不在这里再拆）
    "DG",   # Dentate gyrus
    "FC",   # Fasciola cinerea
    "IG",   # Induseum griseum
]

# RHP: Retrohippocampal region 下到 area 这一层
hpf_rhp_areas = [
    "ENT",   # Entorhinal area
    "PAR",   # Parasubiculum
    "POST",  # Postsubiculum
    "PRE",   # Presubiculum
    "SUB",   # Subiculum
    "ProS",  # Prosubiculum
    "HATA",  # Hippocampo-amygdalar transition area
    "APr",   # Area postrata
]

# 合并一个总的 HPF list（如果你需要）
hpf = hpf_hip_areas + hpf_rhp_areas
    


olf = [
    "MOB", "AOB", "AON", "TT", "DP", "PIR",
    "NLOT", "COA", "PAA", "TR",
]

thalamus_groups = {
    "visual": ["LGd", "LP", "SGN", "POL", "IGL"],
    "somatosensory": ["VPL", "VPLpc", "VPM", "VPMpc", "PO"],
    "auditory": ["MG", "SGN"],
    "motor": ["VAL", "VM"],
    "limbic": ["AV", "AD", "AM", "LD", "PVT", "PT"],
    "prefrontal_midline": ["MD", "IMD", "IAD", "IAM", "PR", "RE", "RH", "SMT"],
    "intralaminar_association": ["CM", "PCN", "CL", "PF", "SPFm", "SPFp", "SPA", "PP"],
    "reticular": ["RT"],
}
all_thalamic_nuclei = sorted({nuc for group in thalamus_groups.values() for nuc in group})

region_list = isocortex + olf + all_thalamic_nuclei + hpf

# ===== 3. 输出目录 =====
out_dir_nz = "plots_NZ_by_region"
out_dir_motif = "plots_motif_real_vs_ER_by_region"
os.makedirs(out_dir_nz, exist_ok=True)
os.makedirs(out_dir_motif, exist_ok=True)

motif_ids = np.arange(1, 14)
motif_labels = [f"M{i}" for i in motif_ids]

def get_region_data(region):
    """兼容两种结构：
    1) motif_results[region][thr] = {...}
    2) motif_results[region] 里直接有 'NZ' / 'motif_count' / 'er_mu'
    """
    if region not in motif_results:
        return None

    data = motif_results[region]
    # 有 threshold 这一层
    if isinstance(data, dict) and "NZ" not in data:
        if thr_to_use not in data:
            return None
        return data[thr_to_use]
    # 没有 threshold 这一层
    return data

# ===== 4. 循环画图 =====
for region in region_list:
    d = get_region_data(region)
    if d is None:
        print(f"[WARN] region {region} not found in motif_results, skip.")
        continue

    NZ = np.asarray(d["NZ"], dtype=float)              # (13,)
    real = np.asarray(d["motif_count"], dtype=float)   # (13,)
    mu = np.asarray(d["er_mu"], dtype=float)           # (13,)
    diff = real - mu

    # ---- (1) NZ-score 柱状图 ----
    fig1, ax1 = plt.subplots(figsize=(6, 4))
    ax1.bar(motif_ids, NZ)
    ax1.set_xticks(motif_ids)
    ax1.set_xticklabels(motif_labels, rotation=0)
    ax1.set_xlabel("Motif ID")
    ax1.set_ylabel("NZ-score")
    title_thr = f", thr={thr_to_use}" if threshold_set is not None else ""
    ax1.set_title(f"{region} NZ-score{title_thr}")
    fig1.tight_layout()
    # fig1.savefig(os.path.join(out_dir_nz, f"{region}_NZ.png"), dpi=300)
    # plt.close(fig1)

    # ---- (2) real / ER / diff 柱状图 ----
    x = np.arange(len(motif_ids))
    width = 0.25

    # 为避免 log2(0)，加一个很小的 eps
    eps = 1e-6
    log2_real = np.log2(real + eps)
    log2_mu   = np.log2(mu   + eps)
    log2_diff = log2_real - log2_mu   # = log2(real/mu)

    fig2, ax2 = plt.subplots(figsize=(7, 4))

    ax2.bar(x - width, log2_real, width, label="log2(Real motif)")
    ax2.bar(x,         log2_mu,   width, label="log2(ER motif)")
    ax2.bar(x + width, log2_diff, width, label="log2(Real) - log2(ER)")

    ax2.set_xticks(x)
    ax2.set_xticklabels(motif_labels, rotation=45)
    ax2.set_xlabel("Motif ID")
    ax2.set_ylabel("log2(count) / log2 ratio")
    ax2.set_title(f"{region} log2 real vs ER motif{title_thr}")
    ax2.legend(frameon=False)

    fig2.tight_layout()
    # fig2.savefig(os.path.join(out_dir_motif, f"{region}_real_ER_diff_log2.png"), dpi=300)
    # plt.close(fig2)

    print(f"[OK] saved plots for {region}")

print("Done. Figures are in:")
print("  -", out_dir_nz)
print("  -", out_dir_motif)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import umap  # pip install umap-learn
import pickle

with open("motif_results_by_region_from_sc.pkl", "rb") as f:
    motif_results = pickle.load(f)


# motif_results = pack["motif_results"]
# === 0. 选一个 threshold ===
thr_to_use = 5   # 或者 threshold_set[0]，看你哪一个是 5um 对应的阈值

# 如果你只想用某一批脑区，就把 use_regions 换成你前面定义的 region_list
# use_regions = region_list
use_regions = sorted(motif_results.keys())

# === 1. 从 motif_results 里收集 NZ 或 Z 向量 ===
region_names = []
nz_list = []

for region in use_regions:
    if region not in motif_results:
        continue
    d_thr = motif_results[region].get(thr_to_use, None)
    if region== "SSp":
        continue

    if d_thr is None:
        # 这个脑区在这个阈值下没有结果（比如报错/跳过）
        continue

    # 用 NZ（归一化 Z）还是 Z，看你需求，这里先用 NZ
    NZ = np.asarray(d_thr["NZ"], dtype=float)  # 形状 (13,)
    # 如果想用 Z：Z_vec = np.asarray(d_thr["Z"], dtype=float)

    region_names.append(region)
    nz_list.append(NZ)

NZ_mat = np.vstack(nz_list)   # 形状：(n_regions, 13)
motif_ids = np.arange(1, 14)
print("参与聚类的脑区数:", len(region_names))

In [ ]:
# === 2. UMAP 降维到 2D（在 NZ 空间上） ===
X = NZ_mat   # 也可以换成 Z_mat = stack([d["Z"]...]) 看你想用哪一个

reducer = umap.UMAP(
    n_neighbors=15,      # 你可以试 5~15 微调
    min_dist=0.2,
    metric="cosine",
    random_state=42,
)
X_umap = reducer.fit_transform(X)   # shape: (n_regions, 2)

# 可选：先看一下 UMAP 散点
plt.figure(figsize=(5, 5))
plt.scatter(X_umap[:, 0], X_umap[:, 1])
for i, name in enumerate(region_names):
    plt.text(X_umap[i, 0], X_umap[i, 1], name, fontsize=7)
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.title(f"UMAP of NZ profiles (thr={thr_to_use})")
plt.tight_layout()
plt.show()